# Choosing the instruments and moods of MTSQ

The first part is for the instruments, the second part for the moods.

How the instruments added by MusicGen (`triplets_instrumentation_add.py`) are chosen, from the instrument tags of the Essentia multilabel classifier:

1. The 40 Essentia instrument tags are grouped into 5 families: **brass, woodwind, guitar, piano and strings** (`instrument_families`).
   Saxophone belongs to brass and to woodwind at the same time, given its timbre similarity with the brass instruments.
2. For every tag we compute how many tracks it is classified in, above the thresholds 0.05, 0.1, 0.2 and 0.3, over the 55,701 tracks of the collection.
3. In every family we select the **most classified tag above 0.1**.
4. Manual decision for the guitar family: `guitar` is generic and `electricguitar` spans a broader set of timbres than `acousticguitar`, so `acousticguitar` is used as the final tag of the family.

Run in the `musicgen` kernel (the families are imported from `triplets_instrumentation_add.py`).

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
ROOT_FOLDER = Path(os.getenv("ROOT_FOLDER"))
CLASSIFICATIONS_FOLDER = Path(os.getenv("CLASSIFICATIONS_FOLDER"))

sys.path.append(str(ROOT_FOLDER / "create_dataset" / "MTSQ"))
from triplets_instrumentation_add import instrument_families

#------------------------------------------ parameters ------------------------------------------#
THRESHOLDS = [0.05, 0.1, 0.2, 0.3]
SELECTION_THRESHOLD = 0.1       #the family tag with most tracks above this activation is selected

#name of each family in instrument_families
FAMILY_NAMES = {'brass': 'brass', 'woodwind': 'woodwind', 'keyboards': 'piano', 'guitars': 'guitar', 'strings': 'strings'}

#manual decisions, applied over the most classified tag of the family: family -> (final tag, reason)
OVERRIDES = {
    'guitar': ('acousticguitar', "'guitar' is generic and 'electricguitar' spans a broader set of timbres than 'acousticguitar'"),
}

#name of the tag in the prompt of the generation, when it is not the name of the tag
PROMPT_NAME = {'acousticguitar': 'acoustic guitar'}

#instruments currently generated by triplets_instrumentation_add.py
CURRENT_INSTRUMENTS = ['saxophone', 'flute', 'piano', 'acoustic guitar', 'violin']

OUT = Path.cwd()

## Instrument activations of the collection

`song_labels_multilabel_original.csv` has the activations of all the 40 instrument tags (the other file has no `voice`) and the 56 mood/theme tags, for the 55,701 tracks.

In [ ]:
labels_df = pd.read_csv(CLASSIFICATIONS_FOLDER / "song_labels_multilabel_original.csv", index_col=0)

#the instrument tags come first, then the mood/theme tags (the first of them is 'action')
instrument_df = labels_df[labels_df.columns[:labels_df.columns.get_loc('action')]]
print(f"{len(instrument_df)} tracks, {instrument_df.shape[1]} instrument tags (from '{instrument_df.columns[0]}' to '{instrument_df.columns[-1]}')")

ungrouped = [tag for tag in instrument_df.columns if not any(tag in tags for tags in instrument_families.values())]
print(f"{len(ungrouped)} tags are in no family: {ungrouped}")

## Frequency of every tag

For each tag: the mean activation, the number of tracks above each threshold (`n>t`) and the fraction of the tracks (`>t`).
Sorted by the fraction above the selection threshold.

In [ ]:
fractions = pd.DataFrame({f">{t}": (instrument_df > t).mean() for t in THRESHOLDS})
counts = pd.DataFrame({f"n>{t}": (instrument_df > t).sum() for t in THRESHOLDS})

frequencies = pd.concat([instrument_df.mean().rename('mean_activation'), fractions, counts], axis=1)
frequencies.insert(0, 'families', [', '.join(FAMILY_NAMES[f] for f, tags in instrument_families.items() if tag in tags) or '-'
                                   for tag in frequencies.index])
frequencies = frequencies.sort_values(f">{SELECTION_THRESHOLD}", ascending=False)
frequencies.index.name = 'tag'
frequencies.to_csv(OUT / "choosing_tags_instrument_frequencies.csv")

pd.set_option('display.max_rows', 100, 'display.width', 200)
display(frequencies.round(3))

In [ ]:
#fraction of the tracks above each threshold, for all the tags, colored by family
family_colors = {'brass': 'tab:orange', 'woodwind': 'tab:green', 'piano': 'tab:blue', 'guitar': 'tab:red', 'strings': 'tab:purple', '-': 'lightgray'}
colors = [family_colors.get(f, 'tab:olive') for f in frequencies['families']]    #tags in two families (saxophone) are olive

fig, axes = plt.subplots(1, len(THRESHOLDS), figsize=(4 * len(THRESHOLDS), 10), sharey=True)
for ax, t in zip(axes, THRESHOLDS):
    ax.barh(frequencies.index, frequencies[f">{t}"], color=colors)
    ax.set_title(f"tracks above {t}")
    ax.set_xlim(0, 1)
axes[0].invert_yaxis()

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in list(family_colors.values()) + ['tab:olive']]
fig.legend(handles, list(family_colors.keys()) + ['brass + woodwind'], loc='lower center', ncol=7)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.savefig(OUT / "choosing_tags_instrument_frequencies.png", dpi=120)
plt.show()

## Selection by family

For every family, its tags sorted by the fraction of tracks above the selection threshold. The first one is the most classified tag; the manual decisions in `OVERRIDES` are applied afterwards.

In [ ]:
rows = []
for family_key, tags in instrument_families.items():
    family = FAMILY_NAMES[family_key]

    family_table = frequencies.loc[tags].drop(columns='families').sort_values(f">{SELECTION_THRESHOLD}", ascending=False)
    print(f"\n=== {family} ===")
    display(family_table.round(3))

    most_classified = family_table.index[0]
    final_tag, reason = OVERRIDES.get(family, (most_classified, ''))

    rows.append({
        'family': family,
        'most_classified_tag': most_classified,
        f"fraction_>{SELECTION_THRESHOLD}": family_table.loc[most_classified, f">{SELECTION_THRESHOLD}"],
        'final_tag': final_tag,
        'final_prompt_name': PROMPT_NAME.get(final_tag, final_tag),
        'manual_decision': reason,
    })

selection = pd.DataFrame(rows).set_index('family')
selection.to_csv(OUT / "choosing_tags_instrument_selection.csv")
print("\n=== selection ===")
display(selection)

selected = selection['final_prompt_name'].tolist()
print("selected instruments:", selected)
print("same as the instruments currently generated:", sorted(selected) == sorted(CURRENT_INSTRUMENTS))

# Choosing the moods of MTSQ

**Part 1** repeats the analysis of the instruments for the mood/theme tags: the frequency of every tag and the selection by family, with the families as they are in `triplets_mood.py`.
Moods are generated only for instrumental songs (MusicGen cannot generate vocals), so the frequencies are computed over the instrumental tracks.

**Part 2** does not assume any family. Every mood/theme tag is used as the prompt on its own, the generated audio is scored, the tags MusicGen renders reliably are kept,
and the families are built from the kept tags using the confusions of the classifier.

In [ ]:
from triplets_mood import mood_families, get_song_tags

MOOD_POPULATION = 'instrumental'     #'instrumental' or 'all': tracks used for the real frequencies (and the base rates of part 2)
INSTRUMENTAL_CONFIDENCE = 0.6        #same threshold as triplets_mood.py

binary_df = pd.read_csv(CLASSIFICATIONS_FOLDER / "song_labels_binary.csv", index_col=0)
instrumental_songs = binary_df[(binary_df['voice_instrumental'] == 'instrumental') &
                               (binary_df['confidence_voice_instrumental'] > INSTRUMENTAL_CONFIDENCE)].index.tolist()

#the mood/theme tags come after the instrument tags, from 'action' to the end
mood_df = labels_df[labels_df.columns[labels_df.columns.get_loc('action'):]]
mood_labels = mood_df.columns.tolist()
real_mood_df = mood_df.loc[instrumental_songs] if MOOD_POPULATION == 'instrumental' else mood_df

print(f"{len(mood_labels)} mood/theme tags, frequencies over {len(real_mood_df)} {MOOD_POPULATION} tracks")
ungrouped_moods = [tag for tag in mood_labels if not any(tag in tags for tags in mood_families.values())]
print(f"{len(ungrouped_moods)} tags are in no family: {ungrouped_moods}")

## Part 1. Frequency of every mood/theme tag

Same columns as for the instruments, sorted by the fraction of tracks above the selection threshold.

In [ ]:
mood_fractions = pd.DataFrame({f">{t}": (real_mood_df > t).mean() for t in THRESHOLDS})
mood_counts = pd.DataFrame({f"n>{t}": (real_mood_df > t).sum() for t in THRESHOLDS})

mood_frequencies = pd.concat([real_mood_df.mean().rename('mean_activation'), mood_fractions, mood_counts], axis=1)
mood_frequencies.insert(0, 'families', [', '.join(f for f, tags in mood_families.items() if tag in tags) or '-'
                                        for tag in mood_frequencies.index])
mood_frequencies = mood_frequencies.sort_values(f">{SELECTION_THRESHOLD}", ascending=False)
mood_frequencies.index.name = 'tag'
mood_frequencies.to_csv(OUT / "choosing_tags_mood_frequencies.csv")

display(mood_frequencies.round(3))

In [ ]:
#fraction of the tracks above each threshold, for all the tags, colored by family
mood_family_colors = dict(zip(mood_families.keys(), plt.cm.tab10.colors))
mood_family_colors['-'] = 'lightgray'
mood_colors = [mood_family_colors.get(f, 'black') for f in mood_frequencies['families']]    #tags in two families are black

fig, axes = plt.subplots(1, len(THRESHOLDS), figsize=(4 * len(THRESHOLDS), 12), sharey=True)
for ax, t in zip(axes, THRESHOLDS):
    ax.barh(mood_frequencies.index, mood_frequencies[f">{t}"], color=mood_colors)
    ax.set_title(f"tracks above {t}")
    ax.set_xlim(0, 1)
axes[0].invert_yaxis()

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in list(mood_family_colors.values()) + ['black']]
fig.legend(handles, list(mood_family_colors.keys()) + ['two families'], loc='lower center', ncol=5)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUT / "choosing_tags_mood_frequencies.png", dpi=120)
plt.show()

## Part 1. Selection by family

For every family of `triplets_mood.py`, its tags sorted by the fraction of tracks above the selection threshold. `current_prompt_word` is the word used in the generation prompt (the name of the family).

In [ ]:
rows = []
for family, tags in mood_families.items():
    family_table = mood_frequencies.loc[tags].drop(columns='families').sort_values(f">{SELECTION_THRESHOLD}", ascending=False)
    print(f"\n=== {family} ===")
    display(family_table.round(3))

    most_classified = family_table.index[0]
    rows.append({
        'family': family,
        'current_prompt_word': family,
        'most_classified_tag': most_classified,
        f"fraction_>{SELECTION_THRESHOLD}": family_table.loc[most_classified, f">{SELECTION_THRESHOLD}"],
        f"fraction_>{SELECTION_THRESHOLD}_of_prompt_word": family_table.loc[family, f">{SELECTION_THRESHOLD}"],
        'prompt_word_is_most_classified': most_classified == family,
    })

mood_selection = pd.DataFrame(rows).set_index('family')
mood_selection.to_csv(OUT / "choosing_tags_mood_selection.csv")
print("\n=== selection ===")
display(mood_selection)

# Part 2. Prompting every tag on its own

Every candidate tag is used as the mood of the prompt of `triplets_mood.py` (`A {tag} {genre} {subgenre} song with only {instruments}`), for random instrumental songs.
The generated audio is scored with the Essentia mood/theme classifier. A tag is **reliable** when the clips prompted with it score high in *that tag*, compared with the clips prompted with the other tags.

For every tag `t`, with the clips prompted with `t` as the *own* clips and all the other clips as the *other* clips:
- `auc`: probability that an own clip has a higher activation of `t` than an other clip (0.5 = the prompt has no effect, 1 = always higher). It does not depend on any activation threshold.
- `gen>x`, `base>x`, `lift>x`: fraction of own clips with an activation of `t` above `x`, the same fraction in the real tracks, and their ratio.
- `top1`, `top3`, `mean_rank`: rank of `t` among the 56 activations of the own clips.

Resumable: clips and scores that already exist are skipped. Run in the `musicgen` kernel.

In [ ]:
import time, random
import torch
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

sys.path.append(str(ROOT_FOLDER / "feature_extraction" / "extractors"))
from essentia_classifiers import EssentiaModels

#------------------------------------------ parameters ------------------------------------------#
MOOD_N_SONGS = 10                 #instrumental songs. Quick check first (it prints the time per clip), then increase
SEED = 0
CANDIDATE_TAGS = mood_labels      #all the mood/theme tags; use a shorter list to try only some
MOOD_BATCH_SIZE = 14              #prompts generated at once by MusicGen
MIN_AUC = 0.75                    #a tag is kept as reliable when its auc is at least this
FAMILY_DISTANCE_CUT = 0.5         #cosine distance at which the dendrogram is cut to form the families

CANDIDATES_DIR = OUT / "mood_candidates"
GENERATED_DIR = CANDIDATES_DIR / "generated"
CANDIDATES_DIR.mkdir(exist_ok=True)

print(f"{MOOD_N_SONGS * len(CANDIDATE_TAGS)} clips to generate, about {MOOD_N_SONGS * len(CANDIDATE_TAGS) * 2.5 / 3600:.1f} h at 2.5 s per clip")

In [ ]:
#random instrumental songs with tags to build the prompt, as in triplets_mood.py
candidates = sorted(instrumental_songs)
random.Random(SEED).shuffle(candidates)

genre_df = pd.read_csv(CLASSIFICATIONS_FOLDER / "song_genre.csv").set_index('song')
multilabel_df = pd.read_csv(CLASSIFICATIONS_FOLDER / "song_labels_multilabel.csv", index_col=0).set_index('song')
prompt_instrument_df = multilabel_df[[c for c in multilabel_df.columns if c.startswith('instrument_')]]
prompt_instrument_df.columns = prompt_instrument_df.columns.str.replace('^instrument_', '', regex=True)

candidate_songs, candidate_song_tags = [], {}
for song in candidates:
    if len(candidate_songs) >= MOOD_N_SONGS:
        break
    tags = get_song_tags(prompt_instrument_df.loc[song], None, genre_df.loc[song])
    if tags is not None:
        candidate_songs.append(song)
        candidate_song_tags[song] = tags

pd.Series(candidate_songs, name='song').to_csv(CANDIDATES_DIR / "songs.csv", index=False)

def tag_prompt(tag, tags):
    genre, subgenre, instruments = tags
    return f"A {tag} {genre} {subgenre} song with only {', '.join(instruments)}"

prompts = pd.DataFrame([{'song': song, 'tag': tag, 'prompt': tag_prompt(tag, candidate_song_tags[song])}
                        for song in candidate_songs for tag in CANDIDATE_TAGS])
prompts.to_csv(CANDIDATES_DIR / "prompts.csv", index=False)
print(f"{len(candidate_songs)} songs, {len(prompts)} prompts")

In [ ]:
musicgen = MusicGen.get_pretrained('facebook/musicgen-melody')
musicgen.set_generation_params(duration=10.0)

def candidate_path(tag, song):
    return GENERATED_DIR / tag / f"{song.split('/')[-1]}.wav"

torch.manual_seed(SEED)
n_generated, t0 = 0, time.time()

for i, song in enumerate(candidate_songs):
    todo = [tag for tag in CANDIDATE_TAGS if not candidate_path(tag, song).exists()]

    for start in range(0, len(todo), MOOD_BATCH_SIZE):
        batch = todo[start:start + MOOD_BATCH_SIZE]
        wav = musicgen.generate([tag_prompt(tag, candidate_song_tags[song]) for tag in batch])

        for tag, one_wav in zip(batch, wav):
            path = candidate_path(tag, song)
            path.parent.mkdir(parents=True, exist_ok=True)
            #loudness normalization at -14 LUFS, as in triplets_mood.py
            audio_write(str(path.with_suffix('')), one_wav.cpu(), musicgen.sample_rate, strategy="loudness", loudness_compressor=True)
        n_generated += len(batch)

    if todo:
        print(f"song {i + 1}/{len(candidate_songs)}: {n_generated} clips, {(time.time() - t0) / n_generated:.1f}s per clip", flush=True)

print(f"done: {n_generated} new clips in {time.time() - t0:.0f}s")

In [ ]:
essentia = EssentiaModels(os.getenv("WEIGHTS_PATH"))

scores_path = CANDIDATES_DIR / "scores.csv"
done = pd.read_csv(scores_path) if scores_path.exists() else pd.DataFrame()
done_paths = set(done['path']) if len(done) else set()
rows = done.to_dict('records')

t0, n_new = time.time(), 0
for song in candidate_songs:
    for tag in CANDIDATE_TAGS:
        path = candidate_path(tag, song)
        if str(path) in done_paths or not path.exists():
            continue

        embeddings = essentia.get_embeddings(str(path))
        row = {'tag': tag, 'song': song, 'path': str(path)}
        for label, score in zip(*essentia.get_multiclass_labels(embeddings, 'moodtheme')):
            row[label] = float(score)
        rows.append(row)
        n_new += 1

pd.DataFrame(rows).to_csv(scores_path, index=False)
print(f"{n_new} new clips scored, {len(rows)} in total ({time.time() - t0:.0f}s)")

## Part 2. Is every tag reliably generated?

Sorted by `auc`. The `base>x` columns are the fractions of the real tracks (`MOOD_POPULATION`) above `x`.

In [ ]:
scores = pd.read_csv(CANDIDATES_DIR / "scores.csv")
scores = scores[scores['tag'].isin(CANDIDATE_TAGS)]
activations = scores[mood_labels]
prompted = scores['tag']

ranks_in_clip = activations.rank(axis=1, ascending=False, method='min')     #rank of every tag inside each clip

rows = []
for tag in CANDIDATE_TAGS:
    is_own = (prompted == tag).values
    own, other = activations.loc[is_own, tag], activations.loc[~is_own, tag]

    #auc by the sum of ranks (Mann-Whitney)
    sum_ranks = pd.concat([own, other]).rank().iloc[:len(own)].sum()
    auc = (sum_ranks - len(own) * (len(own) + 1) / 2) / (len(own) * len(other))

    own_ranks = ranks_in_clip.loc[is_own, tag]
    row = {'tag': tag, 'n': len(own), 'mean_own': own.mean(), 'mean_other': other.mean(), 'auc': auc,
           'top1': (own_ranks == 1).mean(), 'top3': (own_ranks <= 3).mean(), 'mean_rank': own_ranks.mean()}
    for t in THRESHOLDS:
        generated, base = (own > t).mean(), (real_mood_df[tag] > t).mean()
        row[f'gen>{t}'], row[f'base>{t}'] = generated, base
        row[f'lift>{t}'] = generated / base if base > 0 else np.nan
    rows.append(row)

reliability = pd.DataFrame(rows).set_index('tag').sort_values('auc', ascending=False)
reliability['reliable'] = reliability['auc'] >= MIN_AUC
reliability.to_csv(CANDIDATES_DIR / "reliability.csv")

display(reliability.round(3))
reliable_tags = reliability.index[reliability['reliable']].tolist()
print(f"\n{len(reliable_tags)} of {len(reliability)} tags are reliable (auc >= {MIN_AUC}): {reliable_tags}")

## Part 2. Families from the confusions

For every reliable tag, the *profile* is the mean activation of the 56 tags over its clips, minus the mean activation over all the clips: what the prompt adds to the generated audio.
Two tags are confused when their profiles are similar (cosine similarity). The tags are grouped by hierarchical clustering (average linkage) cut at `FAMILY_DISTANCE_CUT`.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist

if len(reliable_tags) < 3:
    print("fewer than 3 reliable tags, no families to build")
else:
    overall_mean = activations.mean()
    profiles = pd.DataFrame({tag: activations[(prompted == tag).values].mean() - overall_mean for tag in reliable_tags}).T

    distances = pdist(profiles.values, metric='cosine')
    tree = linkage(distances, method='average')

    fig, ax = plt.subplots(figsize=(14, 5))
    dendrogram(tree, labels=profiles.index.tolist(), leaf_rotation=90, color_threshold=FAMILY_DISTANCE_CUT, ax=ax)
    ax.axhline(FAMILY_DISTANCE_CUT, color='gray', linestyle='--')
    ax.set_ylabel('cosine distance between profiles')
    plt.tight_layout()
    plt.savefig(CANDIDATES_DIR / "families_dendrogram.png", dpi=120)
    plt.show()

    cluster_ids = fcluster(tree, t=FAMILY_DISTANCE_CUT, criterion='distance')
    candidate_families = pd.DataFrame({'tag': profiles.index, 'family_id': cluster_ids,
                                       'auc': reliability.loc[profiles.index, 'auc'].values}).sort_values(['family_id', 'auc'], ascending=[True, False])

    #each family is named after its most reliable tag
    candidate_families['family'] = candidate_families.groupby('family_id')['tag'].transform('first')
    candidate_families.to_csv(CANDIDATES_DIR / "candidate_families.csv", index=False)

    for family, group in candidate_families.groupby('family', sort=False):
        print(f"{family}: " + ", ".join(f"{t} ({a:.2f})" for t, a in zip(group['tag'], group['auc'])))